# Fission-model GA unfolding and Tikhonov with the generalized discrepancy principle (Ogorodnikov 2024)

This notebook demonstrates the two multisphere-spectrometry methods
ported from the review

> I. N. Ogorodnikov, "Inverse problems of spectroscopy and spectrometry
> in applied research", *Траектория исследований* 2 (10), pp. 42-83
> (2024):

* **`unfold_fission_ga`** — the `BonnerFinder()` algorithm (article
  sections 4-5).  The spectrum is parameterized in the FRUIT paradigm
  as a superposition of a thermal Maxwellian, an epithermal `1/E^b`
  tail with cutoff and a Watt-type fast fission component (article
  eq. 4.29) with the seven free parameters `a1, a2, a3, b, beta,
  alpha, TF`.  Stage 1 scans the parameter hypercube globally with a
  genetic (differential-evolution) algorithm minimizing the L1
  discrepancy of the folded readings (eq. 4.32); stage 2 polishes the
  best point with a bounded nonlinear least-squares routine.  The
  article's fit-validation criteria (FOM, per-sphere relative
  uncertainties, residual sign alternation, model-norm window) are
  reported alongside the fit.
* **`unfold_tikhonov_sobolev_dp`** — the `alfaFinder()` regularization
  (article sections 3, 5).  Tikhonov smoothing with the discrete
  Sobolev $W_2^1$ penalty solves the Euler equation
  $A^{*}A\,z + \alpha\,(z - z'') = A^{*}u$ (eq. 3.10), and the
  regularization parameter $\alpha^{*}$ is selected as the root of the
  generalized discrepancy $\rho(\alpha) = \|A z_\alpha - b\|^2 -
  \delta^2$ (eq. 3.8).  The standalone selection routine
  `alpha_finder_generalized_discrepancy` is demonstrated as well.

Both methods run on the built-in GSF Bonner-sphere responses
(`RF_GSF`, 10 spheres `0in`-`18in`, 60 energy bins) in the same
quasi-real experiment scheme the article uses for solver validation:
fold a model spectrum, add a random noise signal (eqs. 4.21-4.22),
reconstruct and compare with the truth — which never enters the
unfolding.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bssunfold import Detector, RF_GSF
from bssunfold.core import (
    FISSION_PARAM_NAMES,
    alpha_finder_generalized_discrepancy,
    fission_model,
    generalized_discrepancy,
)
from bssunfold.core._matrix_utils import compute_log_steps
from bssunfold.core.dose_calculation import calculate_dose_rates
from bssunfold.utils.comparison import compare_spectra

detector = Detector(RF_GSF)
E = detector.E_MeV
names = detector.detector_names
print(f"Detector grid: {detector.n_energy_bins} bins, "
      f"{E[0]:.1e} - {E[-1]:.1f} MeV")
print("Spheres:", ", ".join(names))
detector.plot_response_functions()


## 1. Quasi-real experiment (article eqs. 4.21-4.22)

The ground truth is a Fission-model curve with reference parameters
inside the article's bounds and an overall fluence scale of $10^4$.
Detector readings are the folded responses contaminated with a 1 %
uniform noise signal; $\delta = \|\Delta b\|$ — the norm of that
known noise vector — is the discrepancy level consumed by the
alfaFinder in section 4.  The truth spectrum is stored as per-bin
fluence (model shape times the lethargy width of each bin), the
convention shared by all solvers of the package.


In [ ]:
A = np.array([detector.sensitivities[n] for n in names])
ln_steps = compute_log_steps(E, detector.n_energy_bins) * np.log(10)

true_params = dict(a1=0.35, a2=0.25, a3=0.40, b=0.15,
                   beta=0.35, alpha=0.6, TF=1.4)
phi_scale = 1.0e4
# Per-bin fluence = model shape * lethargy width * overall scale
spectrum_true = fission_model(E, **true_params) * ln_steps * phi_scale

b_exact = A @ spectrum_true
noise_level = 0.01
rng = np.random.default_rng(42)
b_noisy = b_exact + noise_level * np.abs(b_exact) * rng.uniform(
    -1.0, 1.0, b_exact.shape)
readings = {n: float(v) for n, v in zip(names, b_noisy)}
delta = float(np.linalg.norm(b_noisy - b_exact))

print("Effective readings:")
for nm in names:
    print(f"  {nm:>5s}: {readings[nm]:.4g}")
print(f"discrepancy level delta = {delta:.4g}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.loglog(E, spectrum_true / ln_steps, "k-", lw=1.5)
ax.set(xlabel="E, MeV", ylabel="fluence per lethargy bin, a.u.",
       title="Fission-model truth (quasi-real experiment)")
ax.grid(True, which="both", ls=":", alpha=0.5)

ax = axes[1]
ax.bar(np.arange(len(names)), [readings[nm] for nm in names],
       color="steelblue")
ax.set_yscale("log")
ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=45)
ax.set(xlabel="sphere", ylabel="reading, a.u.",
       title="Effective Bonner-sphere readings")
ax.grid(True, axis="y", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 2. Fission-model GA unfolding (`unfold_fission_ga`)

Stage 1 (`scipy.optimize.differential_evolution` — the genetic
algorithm, analogue of the article's SciLab `optim_ga`) searches the
7-parameter hypercube globally, minimizing the L1 discrepancy between
folded and measured readings (article eq. 4.32).  Stage 2
(`scipy.optimize.least_squares`, bounded `trf` — the analogue of
SciLab `leastsq`) polishes the best genome.  Because the packaged GSF
responses are absolutely calibrated, an eighth free parameter — the
overall scale `phi_scale`, log10-parameterized — matches the absolute
level (`fit_scale=True`, the default).

Note on identifiability: the spectrum depends on the *weight
fractions* $a_i/\\sum_i a_i$ and on the product
$(\\sum_i a_i)\\cdot\\phi_{\\mathrm{scale}}$ rather than on the
individual weights, so the parameterization is redundant by
construction; with noisy readings the shape parameters become
correlated as well.  The article therefore judges the fit by
reading-level criteria (section 3 below), not by parameter
uniqueness.


In [ ]:
result_ga = detector.unfold_fission_ga(readings, ga_maxiter=80,
                                       random_state=3)

fitted = result_ga["model_params"]
print(f"method: {result_ga['method']}  "
      f"(fit_scale={result_ga['fit_scale']}, "
      f"LM={result_ga['lm_method']})")
print("fitted Fission-model parameters:")
for key in FISSION_PARAM_NAMES:
    print(f"  {key:>5s} = {fitted[key]:7.4f}")
wf = fitted["weight_fractions"]
print(f"  weight fractions a1/a2/a3 = "
      f"{wf['a1']:.3f} / {wf['a2']:.3f} / {wf['a3']:.3f}")
print("  (true fractions: 0.350 / 0.250 / 0.400; recovered in the "
      "noiseless limit,")
print("   with 1 % noise the parameters trade off along correlated "
      "directions)")

rel_resid = result_ga["residual_norm"] / np.linalg.norm(b_noisy)
print(f"relative folded residual: {rel_resid:.3e}")

quality = compare_spectra(
    result_ga["spectrum"], spectrum_true,
    metrics=["pearson_r", "relative_flux_error"],
)
print(f"pearson_r = {quality['pearson_r']:.4f}   "
      f"relative_flux_error = {quality['relative_flux_error']:.2%}")

dose_ref = calculate_dose_rates(spectrum_true, detector.cc_icrp116)
dose_ga = result_ga["doserates"]
print("dose-rate deviation (GA vs truth): "
      + ", ".join(f"{k} {dose_ga[k] / dose_ref[k] - 1:+.2%}"
                  for k in ("AP", "ROT", "ISO")))

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(E, spectrum_true / ln_steps, "k-", lw=2,
          label="Fission-model truth")
ax.loglog(E, result_ga["spectrum"] / ln_steps, "C1-o", ms=3, lw=1,
          label="unfold_fission_ga")
ax.set(xlabel="E, MeV", ylabel="fluence per lethargy bin, a.u.",
       title="Fission-model GA unfolding (quasi-real experiment)")
ax.set_xlim(E[0], 30)
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()


## 3. The article's fit-validation criteria

The article validates a multisphere fit by

* the per-sphere relative uncertainties
  $\varepsilon_m = |N_m^{\mathrm{calc}} - N_m^{\mathrm{exp}}| /
  N_m^{\mathrm{exp}}$ (tenths of a percent for good fits),
* alternation of the residual signs along the sphere series — long
  runs of equal signs indicate model misspecification,
* the figure of merit (FOM),
* and, for normalized problems, the model-spectrum norm within
  $[0.6,\,1.2]$ (enabled with `fit_scale=False`).

All checks are assembled in the `validation` entry of the result.


In [ ]:
validation = result_ga["validation"]
print(f"FOM                        : {validation['fom_percent']:.3f} %")
print(f"max per-sphere uncertainty : "
      f"{validation['max_relative_uncertainty']:.3%}")
print(f"residual sign changes      : "
      f"{validation['residual_sign_changes']} "
      f"(mixed: {validation['signs_mixed']})")
print(f"validation passed          : {validation['passed']}")

resid_rel = (A @ result_ga["spectrum"] - b_noisy) / b_noisy
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(len(names)), resid_rel * 100,
       color=["C2" if r <= 0 else "C3" for r in resid_rel])
ax.axhline(0.0, color="k", lw=0.8)
ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=45)
ax.set(xlabel="sphere", ylabel="relative residual, %",
       title="Per-sphere fit residuals "
             f"({validation['residual_sign_changes']} sign changes)")
ax.grid(True, axis="y", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 4. Tikhonov-Sobolev unfolding with the discrepancy principle (`unfold_tikhonov_sobolev_dp`)

The alfaFinder solves $\min_z \|A z - b\|^2 + \alpha\,\|z\|^2_{W_2^1}$:
the discrete Sobolev penalty is the first-difference operator $L$, so
the Euler equation reads $A^{*}A\,z + \alpha\,L^{*}\!L\,z = A^{*}b$
(article eq. 3.10).  For a known error level $\delta$ the
regularization parameter follows from the generalized discrepancy
principle

$$\rho(\alpha^{*}) = \|A z_{\alpha^{*}} - b\|^2 - \delta^2 = 0
\qquad \text{(article eq. 3.8).}$$

$\rho(\alpha)$ is monotone in $\alpha$, so the root is bracketed on a
log10 grid and refined with Brent's method; the diagnostic statuses
0 / 1 / 2 mirror the article's `IERR` codes (root found; $\delta$ too
small for the data; $\delta$ larger than any achievable misfit).

One expectation to set: ten readings leave a large null space, which
a smoothness penalty alone constrains only weakly — the DP is a
*conservative misfit diagnostic* (it stops exactly at the noise
level), not a shape-recovery guarantee.  Section 5 quantifies this.


In [ ]:
result_dp = detector.unfold_tikhonov_sobolev_dp(
    readings, delta=delta, random_state=1)

print(f"method             : {result_dp['method']}")
print(f"penalty            : {result_dp['penalty']}")
print(f"alpha*             : {result_dp['alpha']:.4g}")
print(f"||A z - b||^2      : {result_dp['residual_sq']:.4g}")
print(f"delta^2            : {delta ** 2:.4g}")
print(f"discrepancy status : {result_dp['discrepancy_status']} "
      f"(0 = root found)")
print(f"DP converged       : {result_dp['dp_converged']}")

quality = compare_spectra(
    result_dp["spectrum"], spectrum_true,
    metrics=["pearson_r", "relative_flux_error"],
)
print(f"pearson_r = {quality['pearson_r']:.4f}   "
      f"relative_flux_error = {quality['relative_flux_error']:.2%}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(E, spectrum_true / ln_steps, "k-", lw=2,
          label="Fission-model truth")
ax.loglog(E, result_dp["spectrum"] / ln_steps, "C0-", lw=1.2,
          label="Tikhonov-Sobolev, DP alpha*")
ax.set(xlabel="E, MeV", ylabel="fluence per lethargy bin, a.u.",
       title="Tikhonov-Sobolev unfolding with the discrepancy principle")
ax.set_xlim(E[0], 30)
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()


## 5. Inside the alpha-finder: the $\rho(\alpha)$ root search

The penalty order is selectable — `sobolev` ($W_2^1$ first
differences), `curvature` ($W_2^2$ second differences) or `identity`
(plain ridge); stiffer penalties bias the reconstruction towards
smoother spectra.  Below: the discrepancy curve for the packaged
responses, the Brent root returned by the standalone
`alpha_finder_generalized_discrepancy`, and a penalty-family
comparison that exposes the null-space effect — fluence leaking
into the weakly constrained high-energy bins, visible in the dose
rates but almost invisible in the folded misfit.


In [ ]:
n_bins = A.shape[1]
L1 = np.diff(np.eye(n_bins), axis=0)          # discrete W_2^1 operator
N_mat = A.T @ A
K_mat = L1.T @ L1
rhs = A.T @ b_noisy

alphas = np.logspace(-6, 6, 41)
rho_grid = np.array([
    generalized_discrepancy(a, N_mat, K_mat, rhs, A, b_noisy, delta ** 2)
    for a in alphas
])

info = alpha_finder_generalized_discrepancy(A, b_noisy, delta)
print(f"alpha*  = {info['alpha']:.6g}")
print(f"rho(a*) = {info['rho']:.3g}   "
      f"({info['n_iter']} iterations, converged = {info['converged']})")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(alphas, rho_grid + delta ** 2, "C0-", lw=1.5,
          label=r"$\|A z_\alpha - b\|^2$")
ax.axhline(delta ** 2, color="k", ls="--", lw=1, label=r"$\delta^2$")
ax.axvline(info["alpha"], color="C3", ls=":", lw=1.5,
           label=rf"$\alpha^* = {info['alpha']:.3g}$")
ax.set(xlabel=r"$\alpha$", ylabel="misfit",
       title="Generalized discrepancy and its root")
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

penalty_family = {}
for pen in ("sobolev", "curvature", "identity"):
    res_pen = detector.unfold_tikhonov_sobolev_dp(
        readings, delta=delta, penalty=pen)
    penalty_family[pen] = res_pen
    q_pen = compare_spectra(res_pen["spectrum"], spectrum_true,
                            metrics=["pearson_r"])
    dose_pen = res_pen["doserates"]
    print(f"{pen:>9s}: alpha* = {res_pen['alpha']:.4g}  "
          f"pearson_r = {q_pen['pearson_r']:.3f}  "
          f"fluence ratio = "
          f"{res_pen['spectrum'].sum() / spectrum_true.sum():.2f}  "
          f"dose AP = {dose_pen['AP'] / dose_ref['AP'] - 1:+.0%}")

print("The folded misfit stays at delta for every penalty, yet the "
      "fluence and dose drift:")
print("the null space of the 10-reading system is filled with "
      "smooth, weakly constrained tails.")
print("Model constraints (section 2) or an energy-range truncation "
      "remove the leak.")


## 6. Package data: IAEA Compendium reference spectra

Finally, both solvers are applied to readings synthesized from the
Monte-Carlo spectra of the
[IAEA Compendium](https://www-nds.iaea.org/benchmarks/) shipped with
the test suite
(`tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv`):

* **Cf-252** (`ISO_ref_Cf252`) is a Watt fission spectrum — exactly
  the shape class the Fission model describes;
* **AmBe** (`ISO_ref_AmBe`) — a fast-neutron source field, unfolded
  with the discrepancy-driven Tikhonov reconstruction (a 2 % noise
  level replaces the explicit $\delta$).

Note that the article's acceptance gate is deliberately strict: the
rigid 7-parameter family misses the two smallest spheres of the
real Monte-Carlo spectrum by more than the 5 % threshold, so
`passed` is False even though the overall folded residual stays
near 3 % — the validation report doing exactly its job of flagging
model-data mismatch.


In [ ]:
reference_csv = pd.read_csv(
    "../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv"
)

readings_cf = detector.get_effective_readings_for_spectra(
    reference_csv[["E_MeV", "ISO_ref_Cf252"]])
res_cf = detector.unfold_fission_ga(readings_cf, ga_maxiter=80,
                                    random_state=3)
rel_cf = res_cf["residual_norm"] / np.linalg.norm(
    list(readings_cf.values()))
print(f"Cf-252 (Fission GA) : folded residual = {rel_cf:.2%}, "
      f"FOM = {res_cf['validation']['fom_percent']:.2f} %, "
      f"passed = {res_cf['validation']['passed']}")

readings_ambe = detector.get_effective_readings_for_spectra(
    reference_csv[["E_MeV", "ISO_ref_AmBe"]])
res_ambe = detector.unfold_tikhonov_sobolev_dp(readings_ambe,
                                               noise_level=0.02)
print(f"AmBe  (Tikhonov-DP) : alpha* = {res_ambe['alpha']:.4g}, "
      f"status = {res_ambe['discrepancy_status']}")

phi_cf = np.interp(E, reference_csv["E_MeV"].values,
                   reference_csv["ISO_ref_Cf252"].values)
phi_ambe = np.interp(E, reference_csv["E_MeV"].values,
                     reference_csv["ISO_ref_AmBe"].values)
q_cf = compare_spectra(res_cf["spectrum"], phi_cf,
                       metrics=["pearson_r"])
q_ambe = compare_spectra(res_ambe["spectrum"], phi_ambe,
                         metrics=["pearson_r"])
print(f"Cf-252 shape vs reference: pearson_r = {q_cf['pearson_r']:.3f}")
print(f"AmBe   shape vs reference: pearson_r = {q_ambe['pearson_r']:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, phi_ref, res, ttl in (
    (axes[0], phi_cf, res_cf, "IAEA Cf-252 — Fission-model GA"),
    (axes[1], phi_ambe, res_ambe, "IAEA AmBe — Tikhonov-Sobolev DP"),
):
    ax.loglog(E, phi_ref, "k-", lw=2, label="IAEA reference")
    ax.loglog(E, res["spectrum"], "C1-o", ms=3, lw=1, label="unfolded")
    ax.set(xlabel="E, MeV",
           ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
           title=ttl)
    ax.set_xlim(E[0], 20)
    ax.grid(True, which="both", ls=":", alpha=0.35)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## Summary

* `unfold_fission_ga` (BonnerFinder) recovers the spectral shape from
  noisy multisphere readings in a two-stage genetic + least-squares
  search: the folded misfit reaches the noise floor, the dose rates
  are reproduced within ~1 %, and the article's validation criteria
  (FOM, per-sphere uncertainties, sign alternation) are reported in
  `result["validation"]`.  The parameterization is redundant, so the
  fit is judged on readings, not on parameter uniqueness.
* `unfold_tikhonov_sobolev_dp` (alfaFinder) selects $\alpha^{*}$
  automatically from the generalized discrepancy
  $\|A z - b\|^2 = \delta^2$; `discrepancy_status = 0` certifies the
  root and `residual_sq` matches $\delta^2$ to the root-finder
  tolerance.  On the underdetermined 10-sphere geometry it acts as a
  conservative misfit diagnostic — smoothness penalties alone leave
  the null space weakly constrained (fluence/dose drift, section 5),
  which is precisely the gap the model-constrained Fission search
  fills.
* The standalone pair `alpha_finder_generalized_discrepancy` /
  `generalized_discrepancy` attaches the same discrepancy-based
  selection to any linear reconstruction and exposes the monotone
  $\rho(\alpha)$ curve.
* On the packaged IAEA Compendium fields, the Cf-252 source is
  reproduced by the Fission family (folded residual ~3 %, pearson
  0.99), while AmBe demonstrates the discrepancy-driven Tikhonov
  reconstruction.
